# GloVe and the Geometry of Word Embeddings

Topic 7 showed that skip-gram with negative sampling (SGNS) fits shifted PMI by a count-weighted logistic PCA. GloVe
fits the same co-occurrence counts directly: weighted least squares on $\log X$, with a bias per word and per
context. This notebook shows what that buys and what it costs. The **weight** decides how far each count is
trusted. The **support restriction** (GloVe only sees pairs that occurred) decides what the vectors cannot see. The
**biases** widen the gauge to affine maps, so the objective fixes the exact parallelograms, which are facts about
count ratios, and not the answers a nearest-neighbor analogy search returns.

Every number is computed by `glove_embedding_geometry.py`, which owns them. The counts are Topic 6's L8 window
matrix on Topic 3's training split of the shared finance corpus, the benchmark is Topic 6's planted sectors (112 dev
and 115 test nouns), and the SGNS model is Topic 7's controlled one (40 epochs, $d = 25$).

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "glove_embedding_geometry.py").exists()
                       else here / "notebooks" / "glove-embedding-geometry"))
import glove_embedding_geometry as gl
ds, w2v = gl.ds, gl.w2v

X = gl.counts_for(gl.CONTEXT, 300)
V = X.shape[0]
print(f"L8: N = {int(X.sum()):,} pairs over {X.nnz:,} observed cells ({X.nnz / V**2:.2%} of {V:,}²)")
print(f"observed cells seen once: {np.mean(X.data == 1):.1%}; at most twice: {np.mean(X.data <= 2):.1%}")
print(f"GloVe's parameters at d = {gl.D}: {V * (2 * gl.D + 2):,}, against {X.nnz:,} observed cells")

## 1. A log-bilinear model of counts

Scores are $s_{ij} = \langle \mathbf{w}_i, \mathbf{c}_j \rangle + b_i + \tilde b_j$, and $\Omega$ is the set of observed cells.

- **Def 1 (GloVe).** $J = \sum_{(i,j)\in\Omega} f(X_{ij})\,(s_{ij} - \log X_{ij})^2$ with
  $f(x) = \min\{1, (x/x_{\max})^\gamma\}$, $x_{\max} = 100$, $\gamma = \tfrac34$. The reported vectors are $W + C$.

**Prop 1 (the ratio argument, with its hypotheses).** If $F > 0$ is continuous and $F(u - v) = F(u)/F(v)$, then
$F(u) = e^{\lambda u}$ (Cauchy's equation). If also every $X_{ij} > 0$ and $F(\langle \mathbf{w}_i - \mathbf{w}_k, \mathbf{c}_j\rangle) = P(j \mid i)/P(j \mid k)$, the scores are
$\log P(j \mid i)$ up to a context constant; a free word bias restores the word–context symmetry the model lacks.

**Prop 2 (Topic 6 Thm 3 iii).** An exact fit on a fully observed table makes
$\mathrm{PMI} = WC^\top + a\mathbf{1}^\top + \mathbf{1}\beta^\top$, so $\operatorname{rank}\mathrm{PMI} \le d + 2$.

In [ ]:
gl.test_ratio_argument()
gl.test_log_bilinear_rank()
gl.test_glove_gradient()
u = np.linspace(-3, 3, 61)
print("Cauchy defect of exp:      ", f"{gl.cauchy_defect(np.exp, u, u):.1e}")
print("Cauchy defect of 1 + u²:   ", f"{gl.cauchy_defect(lambda z: 1 + z**2, u, u):.3g}")

## 2. Three weights

**Thm 1 (GloVe is a tempered, support-restricted Poisson fit).** For $X_{ij} \sim \mathrm{Poisson}(e^{s_{ij}})$, the
excess negative log-likelihood of a cell with $x > 0$ is $x(e^\delta - 1 - \delta)$ with $\delta = s - \log x$: the
Bregman divergence of $e^s$, with curvature $x$ at its minimum. A cell with $x = 0$ costs $e^s$, which has no minimum.
GloVe's loss is, up to a factor of 2, the quadratic $\tfrac12 X\delta^2$ with the weight $X$ replaced by $f(X)$ and the zero cells dropped.

**Cor 1 (three local weights).** Near their optima, Poisson weights a cell by $X$, SGNS by $ab/(a+b)$ (Topic 7 Cor 1),
and GloVe by $f(X)$. SGNS's local loss is GloVe's residual with the biases frozen at $\log k\#(w)$ and
$\log P_\alpha(c)$. Since $f(X)/X$ decreases for every $X$, GloVe's share of weight on rare pairs $\{X \le t\}$ is at least Poisson's,
and strictly larger when there are counts on both sides of $t$.

In [ ]:
gl.test_poisson_bregman()
gl.test_three_weights()
for rows, label in ((False, "all observed cells"), (True, "benchmark nouns' rows")):
    sh = gl.weight_shares(rows)
    print(f"share of each weight by count band, {label}:")
    for name in ("cells", "poisson", "sgns", "glove"):
        print(f"  {name:8}", "  ".join(f"{b:>5}: {v:.3f}" for b, v in zip(w2v.BAND_NAMES, sh[name])))

## 3. What the loss cannot see

**Thm 2 (flat weights and full support).** If every cell is observed and $f \equiv 1$, GloVe's optimum is the truncated
SVD of the double-centered PMI matrix $J_n\,\mathrm{PMI}\,J_m$, plus offsets the biases carry: Topic 6's SVD of a
centered PMI, up to the gauge.

**Prop 3 (support blindness).** $J$ sees only the observed triples. If $\log X$ is additive on $\Omega$ (in particular, if
every observed count is the same), the biases fit everything and the zero vectors are optimal, whatever $\Omega$ is.

In [ ]:
gl.test_flat_weight_closed_form()
gl.test_support_blindness()
rng = np.random.default_rng(0)
Xs = w2v._small_instance(rng, 6, 5) + 1.0
opt = gl.flat_weight_optimum(Xs, 2)
print("Thm 2 on a random 6 × 5 table: minimum", round(opt["loss"], 6), "= Σ_{i>2} σ_i² of", np.round(opt["sv"], 4))

### The corpus: GloVe against SGNS and the SVD of the same counts

All three methods factor the same L8 matrix at $d = 25$, scored on Topic 6's held-out nouns. GloVe uses its published
weight and settings (50 AdaGrad epochs). Most observed cells are seen once, so most of $\log X$ on $\Omega$ is zero, and the biases
explain most of what is left.

In [ ]:
gl.test_benchmark()
b = gl.benchmark_summary()
print(f"{'method':<22}{'dev AUC':>9}{'dev 1-NN':>10}{'test AUC':>10}{'test 1-NN':>11}")
for key, name in (("glove", "GloVe (AdaGrad, W + C)"), ("glove_W", "GloVe (AdaGrad, W)"),
                  ("glove_als", "GloVe (ALS, W + C)"), ("sgns", "SGNS (Topic 7)"), ("svd", "SVD(SPPMI₅), p = 0")):
    print(f"{name:<22}{b[key + '_dev_auc']:>9.3f}{b[key + '_dev_nn']:>10.3f}{b[key + '_test_auc']:>10.3f}{b[key + '_test_nn']:>11.3f}")
print(f"GloVe's test AUC − ½ = {b['glove_test_auc'] - 0.5:+.3f} (jackknife se {b['glove_auc_se']:.3f}); chance 1-NN {b['chance_nn']:.3f}")
print(f"GloVe − SGNS: {b['glove_vs_sgns_auc_diff']:+.3f} AUC (se {b['glove_vs_sgns_auc_diff_se']:.3f})")
print(f"share of the weighted variance of log X on Ω explained by the biases alone: {b['additive_share']:.3f}; "
      f"added by 25-dimensional vectors: {b['vector_share']:.3f}")

### More text, and the pairs never seen

The generator continued to 1,200 and 3,000 documents keeps the same restaurants, so the benchmark nouns keep their
sectors. **Def 2 (imputed zeros)** counts every unseen pair as seen half a time, with GloVe's own weight $f(\tfrac12)$:
a parameter-free way to let GloVe see the support. (Swivel is the published model that does this, with a different
loss.) Both GloVe fits here are by ALS.

In [ ]:
gl.test_scale_curve()
gl.test_zero_imputation()
s = gl.scale_summary()
print(f"{'docs':>6}{'seen once':>11}{'GloVe':>8}{'imputed':>9}{'SVD':>7}   (test AUC)")
for n in gl.SCALE_DOCS:
    print(f"{n:>6}{s[f'singleton_share_{n}']:>11.3f}{s[f'glove_test_auc_{n}']:>8.3f}"
          f"{s[f'imputed_test_auc_{n}']:>9.3f}{s[f'svd_test_auc_{n}']:>7.3f}")

## 4. The algorithm

- **Def 3 (GloVe's AdaGrad, as run).** Each epoch is a fresh permutation of $\Omega$ in minibatches of 512, with the C code's
  per-cell step on $\tfrac12 f(s - \log x)^2$, AdaGrad accumulators starting at 1 and adding the square of each undivided step, and $\eta = 0.05$. Minibatch summation
  replaces the C code's Hogwild updates.
- **ALS**, the cross-check, is exact block descent on $J + \lambda(\lVert W \rVert_F^2 + \lVert C \rVert_F^2)$, $\lambda = 10^{-3}$, biases
  unpenalized: with the contexts fixed, each word's $(\mathbf{w}_i, b_i)$ solves a
  weighted ridge regression, and the objective never increases. Forty sweeps do not converge it either.

In [ ]:
gl.test_training()
g = gl.train_glove(gl.CONTROLLED)
print("AdaGrad J by epoch:", {h["epoch"]: round(h["loss"], 1) for h in g.history})
print("ALS J after 40 sweeps:", round(gl.als_model(300, False).loss, 1))

## 5. Gauge and analogies

**Prop 4 (GloVe's gauge is affine).** Every score is unchanged by $\mathbf{w} \mapsto \mathbf{w}A + t$, $\mathbf{c} \mapsto \mathbf{c}A^{-\top} + s$ with the biases
adjusted, and when $[W, b, \mathbf{1}]$ and $[C, \mathbf{1}, \tilde b]$ have full column rank $d + 2$ for both parameter sets, these
are the only parameter sets with the same scores.

**Prop 5 (ridge).** With a ridge on $W$ and $C$ and free biases, every stationary point is balanced
($W^\top W = C^\top C$) and centered ($\mathbf{1}^\top W = \mathbf{1}^\top C = 0$): given rank $d$ and distinct singular values, Topic 6's $p = \tfrac12$, centered.

- **Def 4 (3CosAdd).** $a : b :: c : {?}$ is answered by $\arg\max_x \cos(\hat x, \hat b - \hat a + \hat c)$ over $x \notin \{a, b, c\}$.

**Thm 3 (exact parallelograms are facts about count ratios).** At an exact fit, with $[C, \mathbf{1}]$ of full column
rank, $\mathbf{w}_b - \mathbf{w}_a = \mathbf{w}_d - \mathbf{w}_c$ iff $\mathrm{PMI}_b - \mathrm{PMI}_a - \mathrm{PMI}_d + \mathrm{PMI}_c$ is
constant across contexts, iff $P(\cdot \mid b)/P(\cdot \mid a) \propto P(\cdot \mid d)/P(\cdot \mid c)$. The condition survives every gauge.

**Cor 2.** 3CosAdd's answer survives per-word scaling and rotations, not a general $A$ or a translation. The
*gauge-free* solver below answers with the tail (other than $b$) whose score rows come closest to a parallelogram, so every
factorization of the same scores gives it the same answers. It chooses among the tails only, so it compares with
3CosAdd within the category.

In [ ]:
gl.test_affine_gauge()
gl.test_ridge_balanced_centered()
gl.test_exact_parallelogram()
gl.test_analogy_gauge_dependence()
gt = gl.gauge_table()
for method, names in (("sgns", ("trained", "p0", "balanced", "p1", "random")),
                      ("glove", ("trained", "balanced", "centered", "translated", "random"))):
    print(f"{method}: variant, score change, test AUC, city → region (3CosAdd open / within / gauge-free)")
    for n in names:
        k = f"{method}|{n}"
        print(f"  {n:<11}{gt[k + '|score_change']:>10.1e}{gt[k + '|test_auc']:>8.3f}   "
              f"{gt[k + '|city_region|open']:.3f} / {gt[k + '|city_region|within']:.3f} / {gt[k + '|city_region|gauge_free']:.3f}")

## 6. Finance case study: a filings knowledge base as offsets

The shared corpus states facts from a fixed knowledge base: each public company's sector, and each headquarters
city's region. If a relation were a constant offset, $\mathbf{w}_\text{Texas} - \mathbf{w}_\text{Houston} \approx
\mathbf{w}_\text{Massachusetts} - \mathbf{w}_\text{Boston}$, and 3CosAdd would find it. TransE (the knowledge-graph topic) makes
$\mathbf{w}_h + \mathbf{r} \approx \mathbf{w}_t$ its training objective instead of hoping a co-occurrence objective yields it.

In [ ]:
gl.test_finance_findings()
f = gl.finance_summary()
for rel in gl.RELATIONS:
    print(f"{gl.RELATION_LABELS[rel]} ({f[f'{rel}|glove|n']} questions; within-category chance {f[f'{rel}|glove|chance_within']:.3f})")
    for name in ("glove", "glove_W", "sgns", "sgns_wc", "svd_p0", "svd_half"):
        k = f"{rel}|{name}"
        print(f"  {name:<9} open {f[k + '|open']:.3f}  within {f[k + '|within']:.3f}  b nearest {f[k + '|b_nearest']:.3f}")
    for name in ("glove", "sgns", "svd"):
        k = f"{rel}|{name}|gauge_free"
        print(f"  gauge-free {name:<5} within {f[k + '_within']:.3f}  defect right {f[k + '_defect_right']:.3f}  wrong {f[k + '_defect_wrong']:.3f}")

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows. Panel A has Topic 7's *revenue* cells (one context per count band,
and one it never meets) with each method's local weight, and a count histogram for recomputing GloVe's band shares.
Panel B has the analogy projections, accuracies, and curated answers for each factorization, and the scale curve.

In [ ]:
gl.test_viz_payload()
P = gl.viz_payload()
for c in P["weights"]["cells"]:
    print(f"revenue·{c['context']:<12} a = {c['a']:>5.0f}   b = {c['b']}   log a = {c['log_a']}   GloVe s = {c['glove_s']}   SGNS x = {c['sgns_x']}")
for rel, cell in P["analogies"].items():
    q = cell["questions"][0]
    print(f"{cell['label']}: {q[0]} : {q[1]} :: {q[2]} : ?  SGNS trained {cell['methods']['sgns']['trained']['answers'][0]}, "
          f"random gauge {cell['methods']['sgns']['random']['answers'][0]}")

In [ ]:
assert gl._run_harness() == 0